# RandomForest vs ExtraTrees — final model comparison
Loads cleaned data, cross-validates both classifiers (5 seeds x 5 folds each, with threshold tuning), trains the RandomForest regressor, then produces **two** submission files — one using RandomForest for validity, one using ExtraTrees — so you can keep both versions.

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve, mean_squared_error, mean_absolute_error

train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')

feature_cols = [
    "Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min",
    "Sensor_S1", "Sensor_S2", "Sensor_S3",
    "S1_missing", "S2_missing", "S3_missing", "S4_missing",
    "is_duplicate_input"
]

X = train[feature_cols]
y_class = train["Validity_Label_enc"]
y_reg = train["Reference_Parameter"]
X_test_final = test[feature_cols]

seeds = [42, 7, 1, 100, 2024]
print(X.shape, X_test_final.shape)

(1000, 12) (350, 12)


## 1. RandomForestClassifier — 5 seeds x 5 folds, default-threshold F1/AUC + tuned threshold per fold
Same rigor as the ExtraTrees check: don't trust one seed.

In [2]:
rf_default_f1_by_seed = []
rf_auc_by_seed = []
rf_best_thresholds = []

for seed in seeds:
    skf_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_f1s, fold_aucs = [], []

    for train_idx, val_idx in skf_seed.split(X, y_class):
        X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

        clf_rf = RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
        )
        clf_rf.fit(X_tr, y_tr)
        probs = clf_rf.predict_proba(X_vl)[:, 1]
        preds = clf_rf.predict(X_vl)

        fold_f1s.append(f1_score(y_vl, preds))
        fold_aucs.append(roc_auc_score(y_vl, probs))

        precisions, recalls, thresholds = precision_recall_curve(y_vl, probs)
        f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
        rf_best_thresholds.append(thresholds[f1s.argmax()])

    rf_default_f1_by_seed.append(np.mean(fold_f1s))
    rf_auc_by_seed.append(np.mean(fold_aucs))
    print(f"Seed {seed}: Mean F1={np.mean(fold_f1s):.4f}, Mean AUC={np.mean(fold_aucs):.4f}")

print(f"\nRF overall avg default-threshold F1: {np.mean(rf_default_f1_by_seed):.4f}")
print(f"RF overall avg AUC: {np.mean(rf_auc_by_seed):.4f}")
print(f"RF mean tuned threshold: {np.mean(rf_best_thresholds):.4f}")
print(f"RF median tuned threshold: {np.median(rf_best_thresholds):.4f}")

Seed 42: Mean F1=0.7135, Mean AUC=0.9977
Seed 7: Mean F1=0.7164, Mean AUC=0.9970
Seed 1: Mean F1=0.7192, Mean AUC=0.9979
Seed 100: Mean F1=0.7531, Mean AUC=0.9980
Seed 2024: Mean F1=0.7509, Mean AUC=0.9939

RF overall avg default-threshold F1: 0.7306
RF overall avg AUC: 0.9969
RF mean tuned threshold: 0.2060
RF median tuned threshold: 0.1867


## 2. ExtraTreesClassifier — same 5 seeds x 5 folds

In [3]:
et_default_f1_by_seed = []
et_auc_by_seed = []
et_best_thresholds = []

for seed in seeds:
    skf_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_f1s, fold_aucs = [], []

    for train_idx, val_idx in skf_seed.split(X, y_class):
        X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

        clf_et = ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
        )
        clf_et.fit(X_tr, y_tr)
        probs = clf_et.predict_proba(X_vl)[:, 1]
        preds = clf_et.predict(X_vl)

        fold_f1s.append(f1_score(y_vl, preds))
        fold_aucs.append(roc_auc_score(y_vl, probs))

        precisions, recalls, thresholds = precision_recall_curve(y_vl, probs)
        f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
        et_best_thresholds.append(thresholds[f1s.argmax()])

    et_default_f1_by_seed.append(np.mean(fold_f1s))
    et_auc_by_seed.append(np.mean(fold_aucs))
    print(f"Seed {seed}: Mean F1={np.mean(fold_f1s):.4f}, Mean AUC={np.mean(fold_aucs):.4f}")

print(f"\nET overall avg default-threshold F1: {np.mean(et_default_f1_by_seed):.4f}")
print(f"ET overall avg AUC: {np.mean(et_auc_by_seed):.4f}")
print(f"ET mean tuned threshold: {np.mean(et_best_thresholds):.4f}")
print(f"ET median tuned threshold: {np.median(et_best_thresholds):.4f}")

Seed 42: Mean F1=0.7237, Mean AUC=1.0000
Seed 7: Mean F1=0.7816, Mean AUC=0.9997
Seed 1: Mean F1=0.7553, Mean AUC=0.9997
Seed 100: Mean F1=0.7754, Mean AUC=0.9999
Seed 2024: Mean F1=0.7793, Mean AUC=0.9993

ET overall avg default-threshold F1: 0.7631
ET overall avg AUC: 0.9997
ET mean tuned threshold: 0.1952
ET median tuned threshold: 0.1967


## 3. Side-by-side comparison

In [4]:
comparison = pd.DataFrame({
    "Model": ["RandomForest", "ExtraTrees"],
    "Avg default-threshold F1": [np.mean(rf_default_f1_by_seed), np.mean(et_default_f1_by_seed)],
    "Avg AUC": [np.mean(rf_auc_by_seed), np.mean(et_auc_by_seed)],
    "Mean tuned threshold": [np.mean(rf_best_thresholds), np.mean(et_best_thresholds)],
    "Median tuned threshold": [np.median(rf_best_thresholds), np.median(et_best_thresholds)]
})
print(comparison.to_string(index=False))

       Model  Avg default-threshold F1  Avg AUC  Mean tuned threshold  Median tuned threshold
RandomForest                  0.730627 0.996879              0.206000                0.186667
  ExtraTrees                  0.763070 0.999720              0.195165                0.196667


## 4. RandomForestRegressor — 5-fold CV (reference only; this is the regressor both submissions will use)

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores, mae_scores = [], []

for train_idx, val_idx in kf.split(X):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_reg.iloc[train_idx], y_reg.iloc[val_idx]

    reg = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    reg.fit(X_tr, y_tr)
    preds = reg.predict(X_vl)

    rmse_scores.append(np.sqrt(mean_squared_error(y_vl, preds)))
    mae_scores.append(mean_absolute_error(y_vl, preds))

print(f"RF Regressor Mean RMSE: {np.mean(rmse_scores):.4f} (+/- {np.std(rmse_scores):.4f})")
print(f"RF Regressor Mean MAE: {np.mean(mae_scores):.4f} (+/- {np.std(mae_scores):.4f})")

RF Regressor Mean RMSE: 2.2166 (+/- 0.6953)
RF Regressor Mean MAE: 0.9588 (+/- 0.1591)


## 5. Train final models on FULL training data

In [6]:
# Final classifiers
final_clf_rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
final_clf_rf.fit(X, y_class)

final_clf_et = ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
final_clf_et.fit(X, y_class)

# Final regressor (shared by both submission versions)
final_reg = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
final_reg.fit(X, y_reg)

print("All three final models trained.")

All three final models trained.


## 6. Generate test predictions for both classifier versions + the shared regressor

In [7]:
# Use the median tuned threshold from the 5-seed search for each model
rf_final_threshold = np.median(rf_best_thresholds)
et_final_threshold = np.median(et_best_thresholds)
print(f"RF threshold: {rf_final_threshold:.4f}")
print(f"ET threshold: {et_final_threshold:.4f}")

# Regressor predictions (same for both submissions)
test_reg_preds = final_reg.predict(X_test_final)

# RF classifier predictions
test_probs_rf = final_clf_rf.predict_proba(X_test_final)[:, 1]
test_labels_rf = pd.Series((test_probs_rf >= rf_final_threshold).astype(int)).map({0: "Valid", 1: "Invalid"})

# ET classifier predictions
test_probs_et = final_clf_et.predict_proba(X_test_final)[:, 1]
test_labels_et = pd.Series((test_probs_et >= et_final_threshold).astype(int)).map({0: "Valid", 1: "Invalid"})

print("RF predicted balance:")
print(test_labels_rf.value_counts(normalize=True))
print("\nET predicted balance:")
print(test_labels_et.value_counts(normalize=True))

RF threshold: 0.1867
ET threshold: 0.1967
RF predicted balance:
Valid      0.871429
Invalid    0.128571
Name: proportion, dtype: float64

ET predicted balance:
Valid      0.871429
Invalid    0.128571
Name: proportion, dtype: float64


## 7. Build both submission files

In [8]:
submission_rf = pd.DataFrame({
    "Test_ID": test["Test_ID"],
    "Predicted_Reference_Parameter": test_reg_preds,
    "Validity_Label": test_labels_rf
})

submission_et = pd.DataFrame({
    "Test_ID": test["Test_ID"],
    "Predicted_Reference_Parameter": test_reg_preds,
    "Validity_Label": test_labels_et
})

print(submission_rf.shape, submission_et.shape)
print(submission_rf.head())
print(submission_et.head())

(350, 3) (350, 3)
    Test_ID  Predicted_Reference_Parameter Validity_Label
0  TST-0278                      32.397544        Invalid
1  TST-0006                      19.157111          Valid
2  TST-0047                      53.208862          Valid
3  TST-0311                      25.528105          Valid
4  TST-0264                      18.511390        Invalid
    Test_ID  Predicted_Reference_Parameter Validity_Label
0  TST-0278                      32.397544        Invalid
1  TST-0006                      19.157111          Valid
2  TST-0047                      53.208862          Valid
3  TST-0311                      25.528105          Valid
4  TST-0264                      18.511390        Invalid


## 8. How much do the two classifier versions actually disagree?

In [9]:
diff_count = (submission_rf["Validity_Label"] != submission_et["Validity_Label"]).sum()
print(f"Rows where RF and ExtraTrees disagree: {diff_count} out of {len(submission_rf)}")

disagreements = test[["Test_ID"]].copy()
disagreements["RF_label"] = submission_rf["Validity_Label"].values
disagreements["ET_label"] = submission_et["Validity_Label"].values
disagreements = disagreements[disagreements["RF_label"] != disagreements["ET_label"]]
print(disagreements)

Rows where RF and ExtraTrees disagree: 2 out of 350
      Test_ID RF_label ET_label
208  TST-0313  Invalid    Valid
253  TST-0063    Valid  Invalid


## 9. Save both versions as separate files

In [13]:
submission_rf.to_csv("submission_randomforest.csv", index=False)
submission_et.to_csv("Freshers_404.csv", index=False)
print("Saved: submission_randomforest.csv and submission_extratrees.csv")

Saved: submission_randomforest.csv and submission_extratrees.csv


##  10. Saving the models

In [11]:
import joblib

joblib.dump(final_clf_et, r"Model/final_classifier_extratrees.pkl")
joblib.dump(final_reg, r"Model/final_regressor_randomforest.pkl")

['final_regressor_randomforest.pkl']

## 11. Loading the models

In [12]:
import joblib

loaded_clf = joblib.load("Model/final_classifier_extratrees.pkl")
loaded_reg = joblib.load("Model/final_regressor_randomforest.pkl")

probs = loaded_clf.predict_proba(X_test_final)[:, 1]
preds = loaded_reg.predict(X_test_final)